# Fine-tuning ResNet-101 para Classificação de Folhas de Soja

Notebook de treinamento do modelo **ResNet-101** com transfer learning para classificação binária (saudável vs. doente).

**Hiperparâmetros ótimos** encontrados via Bayesian Search (Optuna):
- Dropout FC1: 0.4720
- Dropout FC2: 0.2132
- Neurônios FC1: 256
- Neurônios FC2: 256
- Ativação: ReLU
- Optimizer: SGD (lr=0.00116, momentum=0.9895)
- Batch Size: 128

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

# PyTorch Ignite
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping, ModelCheckpoint
from ignite.metrics import Accuracy, Loss

from dotenv import load_dotenv
load_dotenv()

# =============================================
# CONFIGURAÇÃO — ajuste via .env
# =============================================
DATASET_PATH = os.getenv("DATASET_PATH", "/caminho/para/DADOS-DIVIDIDOS")
PRETRAINED_WEIGHTS = os.getenv("RESNET101_PRETRAINED", "/caminho/para/models/ResNet_101_ImageNet_plant-model-84.pth")
RESULTS_DIR = os.getenv("RESULTS_DIR", "./results/resnet101")
CHECKPOINT_DIR = os.getenv("CHECKPOINT_DIR", "./checkpoints/resnet101")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Hiperparâmetros ótimos (Bayesian Search)
BATCH_SIZE = 128
NUM_EPOCHS = 100
LEARNING_RATE = 0.0011601308953236539
MOMENTUM = 0.9895156617003595
DROPOUT1 = 0.4720140263972287
DROPOUT2 = 0.21316165290099465
FC1_NEURONS = 256
FC2_NEURONS = 256
NUM_CLASSES = 2
FEATURE_EXTRACT = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Data Augmentation e Carregamento do Dataset

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(DATASET_PATH, x), data_transforms[x])
    for x in ['train', 'val', 'test']
}

dataloaders = {
    x: torch.utils.data.DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=4)
    for x in ['train', 'val', 'test']
}

class_names = image_datasets['train'].classes
print(f"Classes: {class_names}")
for split in ['train', 'val', 'test']:
    print(f"  {split}: {len(image_datasets[split])} imagens")

## 2. Carregamento do Modelo e Transfer Learning

In [ ]:
def set_parameter_requires_grad(model, feature_extracting):
    """Congela os parâmetros do modelo para feature extraction."""
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

# Carregar ResNet-101 sem pesos ImageNet originais
model = models.resnet101(pretrained=False)
set_parameter_requires_grad(model, FEATURE_EXTRACT)

# Carregar pesos pré-treinados no PlantVillage
state_dict = torch.load(PRETRAINED_WEIGHTS, map_location=device)
# Remover camada FC original (incompatível com nosso classificador)
del state_dict['fc.weight']
del state_dict['fc.bias']
model.load_state_dict(state_dict, strict=False)

# Substituir classificador com hiperparâmetros ótimos do Bayesian Search
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=DROPOUT1),
    nn.Linear(num_features, FC1_NEURONS),
    nn.ReLU(),
    nn.Dropout(p=DROPOUT2),
    nn.Linear(FC1_NEURONS, FC2_NEURONS),
    nn.ReLU(),
    nn.Linear(FC2_NEURONS, NUM_CLASSES),
)

model = model.to(device)
print(f"Classificador customizado:\n{model.fc}")

## 3. Configuração do Treinamento (PyTorch Ignite)

In [ ]:
# Loss e Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    momentum=MOMENTUM
)

# Scheduler (ReduceLROnPlateau)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# ---- Engines do Ignite ----
def train_step(engine, batch):
    model.train()
    inputs, labels = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item(), outputs, labels

def eval_step(engine, batch):
    model.eval()
    with torch.no_grad():
        inputs, labels = batch[0].to(device), batch[1].to(device)
        outputs = model(inputs)
        return outputs, labels

trainer = Engine(train_step)
evaluator = Engine(eval_step)

# Métricas no evaluator
Accuracy().attach(evaluator, 'accuracy')
Loss(criterion).attach(evaluator, 'loss')

# Histórico de métricas
history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}

print("Engines configuradas.")

## 4. Callbacks (EarlyStopping, ModelCheckpoint, Logging)

In [ ]:
# Função de score para EarlyStopping e ModelCheckpoint
def score_function(engine):
    return engine.state.metrics['accuracy']

# EarlyStopping — para se a accuracy não melhorar em 10 épocas
early_stopping = EarlyStopping(
    patience=10,
    score_function=score_function,
    trainer=trainer
)
evaluator.add_event_handler(Events.COMPLETED, early_stopping)

# ModelCheckpoint — salva o melhor modelo
checkpoint = ModelCheckpoint(
    CHECKPOINT_DIR,
    filename_prefix='best',
    n_saved=1,
    score_function=score_function,
    score_name='accuracy',
    require_empty=False
)
evaluator.add_event_handler(Events.COMPLETED, checkpoint, {'model': model})

# Logging — avaliação no final de cada época
@trainer.on(Events.EPOCH_COMPLETED)
def log_training_results(engine):
    # Loss média do treinamento
    train_loss = np.mean([batch[0] for batch in engine.state.output]) if hasattr(engine.state, 'output') else 0
    
    # Avaliar no conjunto de validação
    evaluator.run(dataloaders['val'])
    metrics = evaluator.state.metrics
    val_acc = metrics['accuracy']
    val_loss = metrics['loss']
    
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    # Scheduler step
    scheduler.step(val_loss)
    
    print(f"Época {engine.state.epoch}/{NUM_EPOCHS} — "
          f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f}")

print("Callbacks configurados.")

## 5. Treinamento

In [ ]:
print(f"Iniciando treinamento: {NUM_EPOCHS} épocas, batch_size={BATCH_SIZE}")
print(f"Optimizer: SGD (lr={LEARNING_RATE:.6f}, momentum={MOMENTUM:.4f})")
print("=" * 60)

trainer.run(dataloaders['train'], max_epochs=NUM_EPOCHS)

print("=" * 60)
print("Treinamento finalizado!")

## 6. Avaliação no Conjunto de Teste

In [ ]:
# Carregar melhor modelo salvo pelo checkpoint
best_model_path = os.path.join(CHECKPOINT_DIR, os.listdir(CHECKPOINT_DIR)[-1])
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

# Avaliar no conjunto de teste
evaluator.run(dataloaders['test'])
test_metrics = evaluator.state.metrics

print(f"\n{'='*40}")
print(f"RESULTADOS NO CONJUNTO DE TESTE")
print(f"{'='*40}")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Loss:     {test_metrics['loss']:.4f}")

## 7. Curvas de Aprendizagem

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['val_loss'], label='Validação', color='#e74c3c')
axes[0].set_title('Loss por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['val_accuracy'], label='Validação', color='#2ecc71')
axes[1].set_title('Accuracy por Época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ResNet-101 — Curvas de Aprendizagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'resnet101_learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Gráfico salvo em: {RESULTS_DIR}/resnet101_learning_curves.png")